# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kinzachoudhry8/my-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Locked Lane: Lane 1:** Predicting Content Decline / Refresh Queue Optimization

**Method Chosen:** Random Forest Classifier

**Why this method fits Lane 1:** Our Week 4 baseline rule struggled because it applied rigid, linear penalties to static features like content age. Organic search decay is inherently non-linear and multi-dimensional—a high content_age_days value is only an urgent decay indicator if paired with high historical exposure (impressions_90d) and a drop in recent activity (comparing clicks_last_30d against clicks_prev_30d). A Random Forest naturally captures these feature interactions and decision boundaries without requiring manual non-linear transformations or feature scaling.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Validation Strategy:** Client-Group Holdout Split (client_id)

**Why this split is honest:** Content pages belonging to the same client_id share site-wide domain authority, technical SEO architecture, and industry search trends. If we perform a naive random split, pages from the same client domain would appear in both training and testing sets, allowing the model to simply memorize client-specific traffic baselines. Grouping our split purely by isolating distinct blocks of client_id guarantees a true, leak-free test of how our model predicts content decay on entirely unseen web domains.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

We train our **Random Forest Classifier** using historical signals, engagement metrics, and rolling performance trends. We then benchmark its ranking performance against our **Week 4 Baseline Score Rule** on the exact same test set (test_df) evaluating **Precision @ Top 50 Queue.**

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [2]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Fetch data directly via your public raw URL
url = "https://raw.githubusercontent.com/kinzachoudhry8/my-ml-internship/main/data/raw/content_refresh_anonymized.csv"
try:
    df = pd.read_csv(url)
except Exception:
    df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 2. Re-encode the exact Baseline Logic from Week 4
max_age = df['content_age_days'].max()
max_imp = df['impressions_90d'].max()
df['baseline_score'] = ((df['impressions_90d'] / max_imp * 0.45) +
                        (df['content_age_days'] / max_age * 0.45) +
                        (df['avg_position'] / 100.0 * 0.10)) * 100

# Define our binary target proxy label (1 if declining, 0 otherwise)
df['target_label'] = df['trend_direction'].apply(lambda x: 1 if x == 'down' else 0)

# 3. Comprehensive Feature Engineering (Isolating rolling trends, engagement, and metadata)
num_features = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'word_count', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d'
]
cat_features = ['competition_level', 'content_type', 'main_intent']

# Handle missing values cleanly across numeric metrics
X_num = df[num_features].fillna(0)

# Safely encode categorical elements
X_cat = df[cat_features].fillna('unknown')
le = LabelEncoder()
for col in cat_features:
    X_cat[col] = le.fit_transform(X_cat[col])

# Combine processed features into a single dataframe
X = pd.concat([X_num, X_cat], axis=1)
y = df['target_label']

# 4. Strict Group Holdout Split by Client ID (80% Train / 20% Test)
unique_clients = df['client_id'].unique()
np.random.seed(42)  # Ensures deterministic splits across runs
train_clients = np.random.choice(unique_clients, size=int(len(unique_clients) * 0.8), replace=False)

train_idx = df['client_id'].isin(train_clients)
test_idx = ~train_idx

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
test_df = df[test_idx].copy()

# 5. Fit our Random Forest Model
rf = RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=5, random_state=42)
rf.fit(X_train, y_train)

# 6. Evaluate and calculate Precision@50 metrics
test_df['model_prob'] = rf.predict_proba(X_test)[:, 1]

top_50_baseline = test_df.sort_values(by='baseline_score', ascending=False).head(50)
top_50_model = test_df.sort_values(by='model_prob', ascending=False).head(50)

p50_baseline = top_50_baseline['target_label'].mean()
p50_model = top_50_model['target_label'].mean()

print("="*40)
print("        MODEL VS BASELINE REPORT")
print("="*40)
metrics_summary = pd.DataFrame({
    "Performance Metric": ["Precision @ Top 50 Queue"],
    "Week 4 Baseline Rule": [f"{p50_baseline:.2%}"],
    "Week 5 Random Forest Model": [f"{p50_model:.2%}"]
})
print(metrics_summary.to_string(index=False))
print("="*40)

# 7. Write run receipt metrics to local and parent outputs directory
for dir_path in ['outputs', '../outputs']:
    os.makedirs(dir_path, exist_ok=True)
    model_metrics = {
        "task_phase": "Build - Week 5 Modeling",
        "lane": "Lane 1 - Predicting Content Decline",
        "precision_at_50_baseline": float(p50_baseline),
        "precision_at_50_model": float(p50_model),
        "model_lift_achieved": bool(p50_model > p50_baseline),
        "features_utilized_count": int(X.shape[1])
    }
    with open(os.path.join(dir_path, 'model_metrics.json'), 'w') as f:
        json.dump(model_metrics, f, indent=4)

print("\n Run snapshot successfully generated locally!")

        MODEL VS BASELINE REPORT
      Performance Metric Week 4 Baseline Rule Week 5 Random Forest Model
Precision @ Top 50 Queue               26.00%                     76.00%

 Run snapshot successfully generated locally!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.